# What does it cost to reindex with a better embedding model?

Your documents are indexed with **Gemma 300M** (`google/embeddinggemma-300m`). Moving to **Nemotron 3 Embed 1B** or **Nemotron 3 Embed 8B** means embedding every document again. This notebook measures, on one GPU, what that reindex costs for each model and how much retrieval quality it buys. It reports the measurements in a form you can scale to your own corpus.

The expensive part of indexing is turning PDFs into text: rendering pages, detecting layout, and running OCR. That work does not depend on the embedding model, so NeMo Retriever Library (NRL) runs it once and saves the result to Parquet. Every reindex starts from the Parquet files:

```text
Pipeline A, once:           PDFs → split → extract → page elements → OCR → rows → Parquet
Pipeline B, once per model: Parquet → embed → LanceDB → fixed query set → recall and nDCG
```

| Step | What happens |
|---|---|
| 1 | Download a small public corpus: [ViDoRe v3 HR](https://huggingface.co/datasets/vidore/vidore_v3_hr) |
| 2 | Pipeline A: extract the documents once and write the rows to Parquet |
| 3 | Describe the extracted data, so you can compare it with your own corpus |
| 4 | Pipeline B: define the reindex graph |
| 5 | Reindex with each model and evaluate it on the same queries |
| 6 | Compare GPU cost with retrieval quality |
| 7 | Scale the cost to your corpus size |

## Before you begin

**Hardware.** The measurements are meant for one NVIDIA RTX PRO 6000 Blackwell Server Edition, the GPU in the AWS `g7e.2xlarge` instance. On Blackwell, NRL runs Nemotron 3 Embed 1B from its NVFP4 checkpoint. Gemma 300M and Nemotron 3 Embed 8B run in BF16. The notebook also runs on older data center GPUs with 48 GB or more, such as the A100, but there the 1B model falls back to BF16 and the timings do not match the RTX PRO 6000 prices. The notebook pins itself to one GPU, so the other GPUs of a multi-GPU server stay free.

**Price.** Costs use the Linux on-demand price of `g7e.2xlarge`, $3.36312 per hour in October 2026. Prices change and differ by region, so check the [EC2 On-Demand pricing page](https://aws.amazon.com/ec2/pricing/on-demand/) and update `GPU_PRICE_USD_PER_HOUR` in the configuration cell. GPU time is always billed in whole hours, rounded up.

**Software.** Install NRL with local GPU inference, then replace PyTorch with the CUDA 13 build, as described in the [NRL README](https://github.com/NVIDIA/NeMo-Retriever/blob/main/nemo_retriever/README.md):

```bash
uv pip install "nemo-retriever[local]" matplotlib ipykernel ipywidgets
uv pip uninstall torch torchvision
uv pip install torch==2.11.0 torchvision==0.26.0 -i https://download.pytorch.org/whl/cu130
```

**Data and models.** Everything comes from Hugging Face. Gemma 300M is gated: accept the Gemma license on the [model page](https://huggingface.co/google/embeddinggemma-300m), then sign in with `hf auth login` or set `HF_TOKEN`. Expect about 25 GB in the Hugging Face cache for the dataset, the extraction models, and the three embedders.

**Runtime.** About 12 minutes end to end on one A100 80 GB once the models are cached. Pipeline A takes about 5 minutes of that and runs only once.

## Configuration

`MODELS` lists the embedders to compare. The first one is your current model and the baseline for every comparison. `nvidia/nemotron-3-embed-1b` is the NRL default embedder; NRL picks its NVFP4 or BF16 checkpoint for the GPU.

In [ ]:
import os

# Pin the notebook to one GPU before any CUDA library is imported.
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")
# Keep model-server logs out of the cell outputs.
os.environ.setdefault("VLLM_LOGGING_LEVEL", "WARNING")
os.environ.setdefault("TRANSFORMERS_VERBOSITY", "error")
os.environ.setdefault("LANCE_LOG", "error")

from pathlib import Path

DATASET = "vidore_v3_hr"
QUERY_LANGUAGE = "english"

WORK_DIR = Path("reindex_cost_workdir").resolve()
PDF_DIR = WORK_DIR / "pdfs"
PARQUET_DIR = WORK_DIR / "extracted"
LANCEDB_URI = str(WORK_DIR / "lancedb")

MODELS = {
    "Gemma 300M": "google/embeddinggemma-300m",
    "Nemotron 3 Embed 1B": "nvidia/nemotron-3-embed-1b",
    "Nemotron 3 Embed 8B": "nvidia/Nemotron-3-Embed-8B-BF16",
}
BASELINE = next(iter(MODELS))

GPU_INSTANCE = "g7e.2xlarge"  # AWS EC2, one NVIDIA RTX PRO 6000 Blackwell Server Edition (96 GB)
GPU_PRICE_USD_PER_HOUR = 3.36312  # Linux on-demand, AWS Pricing Calculator, October 2026

TOP_K = 10
KS = (1, 5, 10)

NRL resolves each model name to a concrete checkpoint. On Blackwell GPUs such as the RTX PRO 6000, Nemotron 3 Embed 1B runs its NVFP4 checkpoint. Nemotron 3 Embed 8B and Gemma 300M are published only in BF16, so they run in BF16 on every GPU.

In [ ]:
import pandas as pd

from nemo_retriever.models import resolve_local_embed_model
from nemo_retriever.models.embed_model_spec import resolve_embed_model_spec

checkpoints = {}
for label, model_name in MODELS.items():
    checkpoint = resolve_local_embed_model(model_name, backend="vllm")
    checkpoints[label] = {
        "checkpoint": checkpoint,
        "weights": resolve_embed_model_spec(checkpoint).quantization or "BF16",
    }
print(pd.DataFrame(checkpoints).T.to_string())

if not resolve_local_embed_model("nvidia/nemotron-3-embed-1b", backend="vllm").endswith("NVFP4"):
    print(
        "\nThis GPU has no native NVFP4 support, so Nemotron 3 Embed 1B runs in BF16. "
        f"The timings will not match {GPU_INSTANCE} prices."
    )

The helpers below run a graph in-process, like `InprocessExecutor`, and record the wall-clock time of every stage. For the embedding stage, `run_graph_timed` first embeds a single row, so model startup is timed apart from the work that grows with your corpus. GPU time is converted to billed hours by rounding up to the next whole hour.

In [ ]:
import math
import time

import pandas as pd

from nemo_retriever.graph import UDFOperator


def _stage_name(node):
    return node.operator.name if isinstance(node.operator, UDFOperator) else node.name


def graph_stages(graph):
    names, node = [], graph.roots[0]
    while node is not None:
        names.append(_stage_name(node))
        node = node.children[0] if node.children else None
    return names


def run_graph_timed(graph, data, *, warmup_stage=None):
    start = time.perf_counter()
    resolved = graph.resolve_for_local_execution()
    timings = [{"stage": "graph setup", "seconds": time.perf_counter() - start, "rows_out": None}]
    node = resolved.roots[0]
    while node is not None:
        name = _stage_name(node)
        if name == warmup_stage:
            start = time.perf_counter()
            node.operator.run(data[data["text"].fillna("").str.strip() != ""].head(1))
            timings.append({"stage": f"{name} (model startup)", "seconds": time.perf_counter() - start, "rows_out": 1})
        start = time.perf_counter()
        data = node.operator.run(data)
        timings.append({"stage": name, "seconds": time.perf_counter() - start, "rows_out": len(data)})
        node = node.children[0] if node.children else None
    return data, pd.DataFrame(timings)


def billed_hours(seconds):
    return math.ceil(seconds / 3600)


def gpu_cost_usd(seconds):
    return billed_hours(seconds) * GPU_PRICE_USD_PER_HOUR

## Step 1: The corpus

ViDoRe v3 HR is a public retrieval benchmark built from 14 EU reports about employment and labor markets (1,110 pages). Each query is labeled with the pages that answer it.

The dataset ships each page as an image. The next cell regroups the pages into one PDF per report, which is what a document store would contain. The rebuilt PDFs have no text layer, so Pipeline A must OCR every page.

In [ ]:
from collections import defaultdict

from datasets import load_dataset

corpus = load_dataset(f"vidore/{DATASET}", "corpus", split="test")

pages_by_doc = defaultdict(list)
for row_index, (doc_id, page) in enumerate(zip(corpus["doc_id"], corpus["page_number_in_doc"])):
    pages_by_doc[doc_id].append((page, row_index))

PDF_DIR.mkdir(parents=True, exist_ok=True)
for doc_id, pages in pages_by_doc.items():
    pdf_path = PDF_DIR / f"{doc_id}.pdf"
    if not pdf_path.exists():
        images = [corpus[row_index]["image"].convert("RGB") for _, row_index in sorted(pages)]
        images[0].save(pdf_path, "PDF", save_all=True, append_images=images[1:], resolution=144)

pdf_paths = sorted(PDF_DIR.glob("*.pdf"))
pd.DataFrame(
    {"document": [p.stem for p in pdf_paths], "pages": [len(pages_by_doc[p.stem]) for p in pdf_paths]}
).sort_values("pages", ascending=False).reset_index(drop=True)

Next, load the fixed evaluation set: the English queries and their relevance labels (qrels). NRL's BEIR helper maps each labeled page to a `"<document>_<page>"` ID, the same `pdf_page` ID that NRL attaches to every retrieval hit. Every model is scored against exactly this set.

In [ ]:
from nemo_retriever.tools.recall.beir import load_beir_dataset

eval_set = load_beir_dataset(
    "vidore_hf",
    dataset_name=DATASET,
    query_language=QUERY_LANGUAGE,
    doc_id_field="pdf_page",
)
query_ids, queries, qrels = eval_set.query_ids, eval_set.queries, eval_set.qrels

print(f"{len(queries)} queries, {sum(len(r) for r in qrels.values())} relevant (query, page) pairs")
print("Example query:", queries[0])

## Step 2: Pipeline A — extract once, write Parquet

Pipeline A is the NRL default PDF extraction graph from `build_graph`, the same graph that `create_ingestor().files(...).extract(method="pdfium_hybrid")` builds. `method="pdfium_hybrid"` reads the native PDF text layer where one exists and OCRs pages that have none.

Two stages follow the extraction:

- `ExplodeContentActor` splits each page into one row per text block, table, chart, and infographic. These are the rows an embedder turns into vectors.
- `ParquetWriterOperator` writes the rows to `PARQUET_DIR`. It leaves out the base64 page and element images, which text embedding does not use.

In [ ]:
import shutil

from nemo_retriever.common.params import ExtractParams
from nemo_retriever.graph import ParquetWriterOperator
from nemo_retriever.graph.ingestor_runtime import build_graph
from nemo_retriever.operators import ExplodeContentActor

shutil.rmtree(PARQUET_DIR, ignore_errors=True)

extract_graph = (
    build_graph(extraction_mode="pdf", extract_params=ExtractParams(method="pdfium_hybrid"))
    >> ExplodeContentActor(modality="text")
    >> ParquetWriterOperator(output_dir=str(PARQUET_DIR), exclude_columns=["page_image", "images"])
)
print(" → ".join(graph_stages(extract_graph)))

pdf_input = pd.DataFrame([{"bytes": p.read_bytes(), "path": str(p)} for p in pdf_paths])
_, extract_timings = run_graph_timed(extract_graph, pdf_input)
extract_timings

## Step 3: What was extracted

These numbers describe the input to every reindex. Compare them with your own corpus: the embedding time in Step 5 grows with the number of rows and the amount of text, not with the number of PDF files. Rows without text, such as blank pages, are not embedded.

In [ ]:
from nemo_retriever.graph import ParquetReaderOperator

extracted = ParquetReaderOperator().run(str(PARQUET_DIR))
has_text = extracted["text"].fillna("").str.strip() != ""
chars = extracted.loc[has_text, "text"].str.len()
N_PAGES = sum(len(pages_by_doc[p.stem]) for p in pdf_paths)
extract_seconds = extract_timings["seconds"].sum()

dataset_stats = pd.Series(
    {
        "documents": len(pdf_paths),
        "pages": N_PAGES,
        "extracted rows": len(extracted),
        "rows with text to embed": int(has_text.sum()),
        "extracted text (million characters)": round(chars.sum() / 1e6, 2),
        "characters per page, average": round(chars.sum() / N_PAGES),
        "characters per row, average": round(chars.mean()),
        "characters per row, median": round(chars.median()),
        "characters per row, 95th percentile": round(chars.quantile(0.95)),
        "Parquet size (MB)": round(sum(f.stat().st_size for f in PARQUET_DIR.glob("*.parquet")) / 1e6, 1),
        "extraction time (minutes)": round(extract_seconds / 60, 1),
    },
    name=DATASET,
)
dataset_stats.to_frame()

The rows by content type show where the text comes from. Tables, charts, and infographics are embedded as text too, from the OCR output.

In [ ]:
extracted[has_text].assign(characters=chars).groupby("_content_type")["characters"].agg(
    rows="count", average_characters="mean", total_characters="sum"
).round(0).sort_values("rows", ascending=False)

## Step 4: Pipeline B — reindex from Parquet

Pipeline B is three stages, and none of them touches a PDF:

1. `ParquetReaderOperator` loads the rows written in Step 2.
2. `_BatchEmbedActor` embeds the text with the chosen model. This is the embedding stage of the NRL default ingestion graph.
3. `IngestVdbOperator` writes one LanceDB table per model. It records the model on the table, and NRL's `Retriever` refuses to query that table with any other model.

In [ ]:
from nemo_retriever.common.params import EmbedParams
from nemo_retriever.operators import _BatchEmbedActor
from nemo_retriever.operators.vdb import IngestVdbOperator


def table_name(label):
    return label.lower().replace(" ", "_")


def build_reindex_graph(model_name, table):
    return (
        ParquetReaderOperator()
        >> _BatchEmbedActor(params=EmbedParams(model_name=model_name))
        >> IngestVdbOperator(
            vdb_op="lancedb",
            vdb_kwargs={
                "uri": LANCEDB_URI,
                "table_name": table,
                "overwrite": True,
                "embedding_model_name": model_name,
                "vector_dim": None,
            },
        )
    )


print(" → ".join(graph_stages(build_reindex_graph(MODELS[BASELINE], table_name(BASELINE)))))

## Step 5: Reindex and evaluate each model

For each model, the next cell runs Pipeline B over the Parquet files, then runs the fixed query set through NRL's `Retriever` and scores the ranked pages with recall@k and nDCG@k. The evaluation is timed too: loading the query embedder (measured with one warm-up query), then embedding every query and searching the index.

Each model's GPU memory is released before the next model loads.

In [ ]:
import gc

import lancedb

from nemo_retriever.graph.retriever import Retriever
from nemo_retriever.tools.recall.beir import build_beir_run_from_hits, compute_beir_metrics

METRIC_NAMES = [f"recall@{k}" for k in KS] + [f"ndcg@{k}" for k in KS]


def evaluate_index(table, model_name):
    retriever = Retriever(
        vdb_kwargs={"uri": LANCEDB_URI, "table_name": table},
        embed_kwargs={"model_name": model_name},
        top_k=TOP_K,
    )
    start = time.perf_counter()
    retriever.queries(queries[:1])
    startup_s = time.perf_counter() - start

    start = time.perf_counter()
    hits = retriever.queries(queries)
    query_s = time.perf_counter() - start

    run = build_beir_run_from_hits(query_ids, hits, doc_id_field="pdf_page")
    all_metrics = compute_beir_metrics(qrels, run, ks=KS)
    return {name: all_metrics[name] for name in METRIC_NAMES}, startup_s, query_s, run


results = {}
for label, model_name in MODELS.items():
    table = table_name(label)
    graph = build_reindex_graph(model_name, table)
    indexed, timings = run_graph_timed(graph, str(PARQUET_DIR), warmup_stage="_BatchEmbedActor")
    del graph, indexed
    gc.collect()

    metrics, startup_s, query_s, run = evaluate_index(table, model_name)
    gc.collect()

    results[label] = {
        "timings": timings,
        "rows_indexed": lancedb.connect(LANCEDB_URI).open_table(table).count_rows(),
        "metrics": metrics,
        "eval_seconds": startup_s + query_s,
        "run": run,
    }
    print(f"{label}: reindex {timings['seconds'].sum():.0f} s, evaluation {startup_s + query_s:.0f} s")

The reindex time of each model, stage by stage. Only the embedding stage depends on the model.

In [ ]:
pd.concat({label: r["timings"].set_index("stage")["seconds"] for label, r in results.items()}, axis=1).round(1)

## Step 6: Reindexing cost against retrieval quality

The cost of a model is the GPU time of Pipeline B plus the evaluation, rounded up to whole hours and multiplied by the on-demand price. `rows indexed` can differ between models because each embedder splits text longer than its input limit into several rows. At 1,110 pages every model finishes well inside one hour, so the billed cost here is the same for all three. Step 7 scales the measured time to a larger corpus, where the difference shows.

In [ ]:
summary = pd.DataFrame(
    {
        label: {
            "rows indexed": r["rows_indexed"],
            "reindex time (min)": r["timings"]["seconds"].sum() / 60,
            "evaluation time (min)": r["eval_seconds"] / 60,
            "GPU-hours": (r["timings"]["seconds"].sum() + r["eval_seconds"]) / 3600,
            "billed GPU-hours": billed_hours(r["timings"]["seconds"].sum() + r["eval_seconds"]),
            "GPU cost (USD)": gpu_cost_usd(r["timings"]["seconds"].sum() + r["eval_seconds"]),
            **r["metrics"],
        }
        for label, r in results.items()
    }
).T
print(f"{GPU_INSTANCE} at ${GPU_PRICE_USD_PER_HOUR:.2f} per hour, {len(queries)} queries")
summary.round(4)

How much each model improves on the baseline, as an absolute change and relative to the baseline score:

In [ ]:
baseline_metrics = pd.Series(results[BASELINE]["metrics"])
gains = pd.DataFrame(
    {
        f"{label} vs {BASELINE}": {
            **{f"{m} change": pd.Series(r["metrics"])[m] - baseline_metrics[m] for m in METRIC_NAMES},
            "nDCG@10 relative change": pd.Series(r["metrics"])["ndcg@10"] / baseline_metrics["ndcg@10"] - 1,
        }
        for label, r in results.items()
        if label != BASELINE
    }
)
gains.round(4)

Averages hide how individual queries move. For each query, compare nDCG@10 with the baseline:

In [ ]:
def per_query_ndcg(run):
    return pd.Series({q: compute_beir_metrics({q: qrels[q]}, run, ks=(10,))["ndcg@10"] for q in query_ids})


baseline_ndcg = per_query_ndcg(results[BASELINE]["run"])
movement = {}
for label, r in results.items():
    if label == BASELINE:
        continue
    delta = per_query_ndcg(r["run"]) - baseline_ndcg
    movement[label] = {
        "improved": int((delta > 0).sum()),
        "unchanged": int((delta == 0).sum()),
        "regressed": int((delta < 0).sum()),
    }
pd.DataFrame(movement)

## Step 7: Scale the cost to your corpus

Set `YOUR_CORPUS_PAGES` to the size of your corpus. The estimate keeps model startup as a fixed cost and scales the rest of the measured reindex time by the page count. It assumes pages that carry about as much text as this corpus (see Step 3); if yours carry twice the text per page, use twice the page count.

Because extraction already ran once, it is not part of any reindex. The last row shows what a full re-extraction of the same corpus would cost, for comparison.

In [ ]:
YOUR_CORPUS_PAGES = 1_000_000

scale = YOUR_CORPUS_PAGES / N_PAGES


def scaled_seconds(timings):
    fixed = timings.loc[timings["stage"].str.endswith("(model startup)"), "seconds"].sum()
    return fixed + (timings["seconds"].sum() - fixed) * scale


rows = {}
for label, r in results.items():
    seconds = scaled_seconds(r["timings"])
    rows[label] = {
        "pages per GPU-hour": YOUR_CORPUS_PAGES / (seconds / 3600),
        "GPU-hours": seconds / 3600,
        "billed GPU-hours": billed_hours(seconds),
        "GPU cost (USD)": gpu_cost_usd(seconds),
        "ndcg@10": r["metrics"]["ndcg@10"],
        "recall@10": r["metrics"]["recall@10"],
    }
at_scale = pd.DataFrame(rows).T
at_scale["extra cost vs baseline (USD)"] = at_scale["GPU cost (USD)"] - at_scale.loc[BASELINE, "GPU cost (USD)"]
at_scale["ndcg@10 gain vs baseline"] = at_scale["ndcg@10"] - at_scale.loc[BASELINE, "ndcg@10"]
at_scale["recall@10 gain vs baseline"] = at_scale["recall@10"] - at_scale.loc[BASELINE, "recall@10"]

extract_at_scale = extract_seconds * scale
at_scale.loc[
    "Re-extraction (not needed)", ["pages per GPU-hour", "GPU-hours", "billed GPU-hours", "GPU cost (USD)"]
] = [
    YOUR_CORPUS_PAGES / (extract_at_scale / 3600),
    extract_at_scale / 3600,
    billed_hours(extract_at_scale),
    gpu_cost_usd(extract_at_scale),
]
print(f"Estimate for {YOUR_CORPUS_PAGES:,} pages on one {GPU_INSTANCE} at ${GPU_PRICE_USD_PER_HOUR:.2f} per hour")
at_scale.round({"pages per GPU-hour": 0, "GPU-hours": 1, "GPU cost (USD)": 2}).round(4)

GPU-hours are the same whether the reindex runs on one GPU or many: eight GPUs finish in about an eighth of the wall-clock time for about the same cost.

The chart puts the estimated cost next to the quality each model delivers.

In [ ]:
import matplotlib.pyplot as plt

models = list(MODELS)
fig, (cost_ax, quality_ax) = plt.subplots(1, 2, figsize=(11, 3.5))
cost_ax.barh(models, at_scale.loc[models, "GPU cost (USD)"], color="#76b900")
cost_ax.set_xlabel("USD")
cost_ax.set_title(f"Reindex cost, {YOUR_CORPUS_PAGES:,} pages")
cost_ax.invert_yaxis()
quality_ax.barh(models, at_scale.loc[models, "ndcg@10"], color="#5d8aa8")
quality_ax.set_xlabel("nDCG@10")
quality_ax.set_title(f"Retrieval quality, {len(queries)} queries")
quality_ax.invert_yaxis()
plt.tight_layout()
plt.show()

## Summary

- **Extraction runs once.** Pipeline A turned the PDFs into rows and saved them to Parquet. No reindex repeats rendering, layout detection, or OCR.
- **A reindex is a three-stage graph.** `ParquetReaderOperator`, `_BatchEmbedActor`, and `IngestVdbOperator` build a new index for any supported embedder.
- **Cost and quality are measured the same way for every model.** Each model is timed on the same GPU, billed in whole hours at the on-demand price, and scored on the same queries and qrels.
- **The result scales.** Step 3 describes the data, and Step 7 turns the measured throughput into GPU-hours and cost for your corpus size.

To evaluate a full RAG pipeline on your own data, refer to [NeMo Retriever RAGAS evaluation](nrl_ragas.ipynb).